# Python 기초 통합 프로젝트
## Part 2. 판매 점검 코드를 함수화하고 결과 저장하기

### Part 1과의 연결

Part 1에서 작성한 판매금액 구간 분류와 배송 확인 거래 검색 코드를 함수로 바꾸어 반복 사용이 가능한 구조로 개선합니다.

### 실무 시나리오

영업관리팀은 매일 새로운 판매 데이터에 같은 점검 기준을 적용합니다. 데이터 담당자는 반복 코드를 함수로 정리하고, 배송 확인 대상 거래를 CSV 파일로 저장하여 담당자에게 전달해야 합니다.

### 과제 목표

- 반복 코드를 재사용 가능한 함수로 작성할 수 있습니다.
- 매개변수, 반환값, 기본값을 사용할 수 있습니다.
- `TypeError`, `ValueError`, `PermissionError` 등 오류 유형을 구분해 처리할 수 있습니다.
- 분석 결과를 CSV 파일로 저장하고 다시 확인할 수 있습니다.
- 심화 문제에서 고액 반품 결과를 JSON으로 저장할 수 있습니다.

### 사용 환경

- 결과물: `.ipynb` 파일 1개
- 필수 생성 파일: `delivery_check_sales.csv`
- 심화 생성 파일: `high_value_return_sales.json`
- 사용 언어: Python 3.X
- 사용 라이브러리: pandas

## 제공 코드. 데이터 불러오기

In [1]:
import pandas as pd

DATA_FILE = "자동차_판매_데이터.csv"

df = pd.read_csv(DATA_FILE)

sale_ids = df["SaleID"].tolist()
sale_amounts = df["FinalSaleAmount"].tolist()
delivery_days = df["DeliveryDays"].tolist()
stock_statuses = df["StockStatus"].tolist()
returned_flags = df["IsReturned"].tolist()

print("전체 거래 수:", len(df))

전체 거래 수: 500


# 문제 1. 판매금액 구간 분류 함수 만들기

## 함수

```python
classify_sales_by_amount(sale_ids, sale_amounts)
```

## 요구사항

1. 두 입력값이 리스트가 아니면 `TypeError`를 발생시킵니다.
2. 두 리스트의 길이가 다르면 `ValueError`를 발생시킵니다.
3. 데이터가 비어 있으면 `ValueError`를 발생시킵니다.
4. 고가·중가·일반 거래의 `SaleID`를 딕셔너리로 반환합니다.
5. 함수를 호출하고 결과를 출력합니다.

In [2]:
# 문제 1 코드를 작성하세요.
def classify_sales_by_amount(sale_ids, sale_amounts):
    if not isinstance(sale_ids, list) or not isinstance(sale_amounts, list):
        raise TypeError("입력값은 반드시 리스트 형태여야 합니다.")
    
    if len(sale_ids) != len(sale_amounts):
        raise ValueError("sale_ids와 sale_amounts의 길이가 일치하지 않습니다.")
        
    if len(sale_ids) == 0:
        raise ValueError("입력 데이터가 비어 있습니다.")

    classified_sales = {
        "고가": [],
        "중가": [],
        "일반": []
    }

    for sale_id, amount in zip(sale_ids, sale_amounts):
        if amount >= 70000000:
            classified_sales["고가"].append(sale_id)
        elif amount >= 40000000:
            classified_sales["중가"].append(sale_id)
        else:
            classified_sales["일반"].append(sale_id)

    return classified_sales

result = classify_sales_by_amount(sale_ids, sale_amounts)

print("고가 거래 수:", len(result["고가"]))
print("고가 거래 앞 5개:", result["고가"][:5])
print()
print("중가 거래 수:", len(result["중가"]))
print("중가 거래 앞 5개:", result["중가"][:5])
print()
print("일반 거래 수:", len(result["일반"]))
print("일반 거래 앞 5개:", result["일반"][:5])

고가 거래 수: 71
고가 거래 앞 5개: ['S2025010064', 'S2025010264', 'S2025010093', 'S2025010135', 'S2025010483']

중가 거래 수: 135
중가 거래 앞 5개: ['S2025010447', 'S2025010207', 'S2025010497', 'S2025010416', 'S2025010335']

일반 거래 수: 294
일반 거래 앞 5개: ['S2025010326', 'S2025010129', 'S2025010270', 'S2025010485', 'S2025010152']


# 문제 2. 배송 확인 거래 검색 함수 만들기

## 함수

```python
find_delivery_check_sales(
    sale_ids,
    delivery_days,
    stock_statuses
)
```

## 요구사항

1. 세 입력값이 리스트가 아니면 `TypeError`를 발생시킵니다.
2. 세 리스트의 길이가 다르면 `ValueError`를 발생시킵니다.
3. 배송일 20일 이상이면서 출고완료가 아닌 거래의 `SaleID`를 반환합니다.
4. 정상 입력과 잘못된 입력을 각각 테스트합니다.
5. `TypeError`와 `ValueError`를 별도의 `except`에서 처리합니다.

In [3]:
# 문제 2 코드를 작성하세요.
def find_delivery_check_sales(sale_ids, delivery_days, stock_statuses):
    if not (isinstance(sale_ids, list) and isinstance(delivery_days, list) and isinstance(stock_statuses, list)):
        raise TypeError("모든 입력값은 리스트 형태여야 합니다.")
        
    if not (len(sale_ids) == len(delivery_days) == len(stock_statuses)):
        raise ValueError("입력 리스트들의 길이가 일치하지 않습니다.")
        
    check_ids = []
    for s_id, days, status in zip(sale_ids, delivery_days, stock_statuses):
        if days >= 20 and status != "출고완료":
            check_ids.append(s_id)
            
    return check_ids

print("=== 1. 정상 데이터 테스트 ===")
try:
    result = find_delivery_check_sales(sale_ids, delivery_days, stock_statuses)
    print("배송 확인 대상 거래 수:", len(result))
    print("배송 확인 대상 SaleID: ", result)
except TypeError as e:
    print(f"TypeError 발생: {e}")
except ValueError as e:
    print(f"ValueError 발생: {e}")
print()

print("=== 2. TypeError 테스트 (잘못된 자료형) ===")
try:
    find_delivery_check_sales("not_a_list", delivery_days, stock_statuses)
except TypeError as e:
    print(f"정상적으로 잡힌 TypeError: {e}")
except ValueError as e:
    print(f"ValueError 발생: {e}")
print()

print("=== 3. ValueError 테스트 (길이 불일치) ===")
try:
    find_delivery_check_sales(sale_ids[:5], delivery_days, stock_statuses)
except TypeError as e:
    print(f"TypeError 발생: {e}")
except ValueError as e:
    print(f"정상적으로 잡힌 ValueError: {e}")

=== 1. 정상 데이터 테스트 ===
배송 확인 대상 거래 수: 48
배송 확인 대상 SaleID:  ['S2025010207', 'S2025010051', 'S2025010165', 'S2025020062', 'S2025020216', 'S2025020065', 'S2025020357', 'S2025030117', 'S2025030095', 'S2025030466', 'S2025040178', 'S2025040452', 'S2025050446', 'S2025050352', 'S2025050100', 'S2025050444', 'S2025050111', 'S2025050493', 'S2025050241', 'S2025050401', 'S2025060310', 'S2025060269', 'S2025060072', 'S2025060295', 'S2025060047', 'S2025060032', 'S2025060374', 'S2025070155', 'S2025070033', 'S2025070457', 'S2025070035', 'S2025070328', 'S2025080070', 'S2025080145', 'S2025080194', 'S2025090350', 'S2025090067', 'S2025100460', 'S2025100316', 'S2025100106', 'S2025110197', 'S2025110222', 'S2025110496', 'S2025110238', 'S2025120500', 'S2025120476', 'S2025120002', 'S2025120340']

=== 2. TypeError 테스트 (잘못된 자료형) ===
정상적으로 잡힌 TypeError: 모든 입력값은 리스트 형태여야 합니다.

=== 3. ValueError 테스트 (길이 불일치) ===
정상적으로 잡힌 ValueError: 입력 리스트들의 길이가 일치하지 않습니다.


# 문제 3. 배송 점검 결과를 CSV로 저장하기

## 요구사항

1. 문제 2에서 반환된 `SaleID`로 원본 DataFrame의 거래를 선택합니다.
2. `delivery_check_sales.csv`로 저장합니다.
3. 파일 인덱스는 저장하지 않습니다.
4. 저장 파일을 다시 불러와 거래 수를 비교합니다.
5. `PermissionError`와 그 외 `OSError`를 구분하여 처리합니다.

In [4]:
# 문제 3 코드를 작성하세요.
CSV_SAVE_PATH = "delivery_check_sales.csv"

delivery_check_df = df[df["SaleID"].isin(result)]

try:
    delivery_check_df.to_csv(CSV_SAVE_PATH, index=False, encoding="utf-8-sig")
    print(f"'{CSV_SAVE_PATH}' 저장 성공!")

    saved_df = pd.read_csv(CSV_SAVE_PATH)
    original_count = len(delivery_check_df)
    saved_count = len(saved_df)

    print()
    print("=== 거래 수 일치 검증 ===")
    print(f"추출된 원본 대상 수: {original_count}건")
    print(f"저장 후 다시 읽은 수: {saved_count}건")
    print(f"일치 여부: {original_count == saved_count}")

except PermissionError as e:
    print(f"[PermissionError] 파일 쓰기 권한이 없거나 파일이 열려 있습니다: {e}")
except OSError as e:
    print(f"[OSError] 파일 입출력 중 시스템 오류가 발생했습니다: {e}")

'delivery_check_sales.csv' 저장 성공!

=== 거래 수 일치 검증 ===
추출된 원본 대상 수: 48건
저장 후 다시 읽은 수: 48건
일치 여부: True


# 심화 문제. 고액 반품 검색 및 JSON 저장

이 문제는 **5점 수준을 위한 선택 문제**입니다.

## 요구사항

1. `find_high_value_returns()` 함수를 작성합니다.
2. `min_amount=70_000_000` 기본값을 사용합니다.
3. 입력 자료형이 잘못되면 `TypeError`, 기준값이 잘못되면 `ValueError`를 발생시킵니다.
4. 고액 반품 거래를 JSON 파일로 저장합니다.
5. 저장한 JSON을 다시 읽어 거래 수를 확인합니다.
6. `PermissionError`, `OSError`, `ValueError`를 구분해 처리합니다.

In [5]:
# 심화 문제 코드를 작성하세요.

# 제출 결과물

| 결과물 | 구분 |
|---|---|
| 판매금액 구간 분류 함수 | 필수 |
| 배송 확인 거래 함수 | 필수 |
| `TypeError`, `ValueError` 구분 처리 | 필수 |
| 배송 확인 CSV 저장 및 재확인 | 필수 |
| `PermissionError`, `OSError` 구분 처리 | 필수 |
| 고액 반품 함수와 JSON 저장 | 심화 |